# Lab Day 2 — DeepWeeds (Kaggle/Colab)
Chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**. Mỗi ô chạy lại được (resume).

## 0. Cài đặt

In [ ]:
import os, sys, platform
REPO_URL = "https://github.com/<user>/<repo>.git"   # TODO: đổi
SUB = "submissions/2A202602796-HoDangPhuc/code"       # thư mục code trong repo
IS_KAGGLE = os.path.exists("/kaggle/working")
ROOT = "/kaggle/working" if IS_KAGGLE else "/content"
os.chdir(ROOT)
if not os.path.exists("lab"):
    !git clone -q {REPO_URL} lab
else:
    !git -C lab pull -q
CODE = f"{ROOT}/lab/{SUB}"
sys.path.insert(0, CODE)
!pip -q install timm openpyxl
import torch, timm
print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU -> bật Accelerator")

In [ ]:
import hashlib
os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"
h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for c in iter(lambda: f.read(1 << 20), b""): h.update(c)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", h.hexdigest()
!unzip -q -n data/images.zip -d data/
B = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for n in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{n}.csv {B}/{n}.csv
!ls data; ls data/images | head -3

In [ ]:
IMAGES_DIR = f"{ROOT}/data/images"   # TODO: sửa nếu ls ở trên cho thấy thư mục khác
LABELS_DIR = f"{ROOT}/data/labels"
OUT = f"{ROOT}/out" if IS_KAGGLE else "/content/drive/MyDrive/deepweeds"
if not IS_KAGGLE:
    from google.colab import drive; drive.mount("/content/drive")
os.makedirs(OUT, exist_ok=True)
import experiments as E, train as Tr, dataset as D
PATHS = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, out_dir=f"{OUT}/runs",
             pred_dir=f"{OUT}/predictions", curves_dir=f"{OUT}/curves", num_workers=4)
def cfg(**kw): return Tr.Config(**{**PATHS, **kw})

## Bước 0 — EDA và kiểm tra pipeline

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
tr, va, te = D.load_split(LABELS_DIR)
info = D.check_split(tr, va, te, IMAGES_DIR)          # giao rỗng, hợp = 17509, file tồn tại, ~60/20/20
cnt = pd.DataFrame({k: v["Label"].value_counts().sort_index() for k, v in zip(["train", "val", "test"], [tr, va, te])})
cnt.index = D.CLASS_NAMES; cnt["total"] = cnt.sum(1); print(cnt)
cnt[["train", "val", "test"]].plot.bar(figsize=(9, 3.5), title="Số ảnh theo lớp (fold 0)"); plt.tight_layout(); plt.show()
print("tỉ lệ lớp lớn/nhỏ:", cnt.total.max() / cnt.total.min())
from PIL import Image
fig, ax = plt.subplots(9, 3, figsize=(7, 20))
for c in range(9):
    for j, f in enumerate(tr[tr.Label == c].Filename.head(3)):
        ax[c, j].imshow(Image.open(f"{IMAGES_DIR}/{f}")); ax[c, j].axis("off")
    ax[c, 0].set_title(D.CLASS_NAMES[c], fontsize=8, loc="left")
plt.tight_layout(); plt.show()

In [ ]:
# Loss đầu ≈ ln 9 và overfit 1 batch (chỉ dùng train)
S = E.sanity(cfg(backbone="resnet50"))
assert abs(S["loss_init"] - S["ln9"]) < 0.6 and S["loss_overfit"] < 0.1, S
# Ảnh sau augmentation (đã giải chuẩn hoá) cùng nhãn
import torch
ds = D.DeepWeedsDataset(tr.sample(8, random_state=0), IMAGES_DIR, D.build_transforms(True, 224, "color"))
m, s = torch.tensor(D.IMAGENET_MEAN)[:, None, None], torch.tensor(D.IMAGENET_STD)[:, None, None]
fig, ax = plt.subplots(1, 8, figsize=(16, 2.5))
for a, (x, y, _) in zip(ax, ds): a.imshow((x * s + m).clamp(0, 1).permute(1, 2, 0)); a.set_title(D.CLASS_NAMES[y], fontsize=7); a.axis("off")
plt.show()

## Bước 1 — Backbone (cùng công thức nền, seed 0)

In [ ]:
BACKBONES = ["resnet50", "resnext50", "convnext_tiny", "deit_small", "swin_tiny", "efficientnet_b0", "mobilenetv3"]
for i, bb in enumerate(BACKBONES, 1):
    E.run_resume(cfg(exp_id=f"B{i:02d}_{bb}", backbone=bb, seed=0))
B = E.collect_runs(PATHS["out_dir"], "B"); B

Chọn backbone đi tiếp (lý do dựa trên số liệu, ghi vào báo cáo):

In [ ]:
BEST = "resnet50"   # TODO sau khi xem bảng B

## Bước 2 — Công thức huấn luyện (mỗi lần khác T00 một yếu tố; 1 seed)

In [ ]:
T = {  # exp_id: (trục, override so với T00)
 "T00": ("nền", {}),
 "T01": ("A khởi tạo", dict(init="frozen")),
 "T02": ("A khởi tạo", dict(init="scratch")),
 "T03": ("B augmentation", dict(aug="color")),
 "T04": ("B augmentation", dict(aug="trivial")),
 "T05": ("B augmentation", dict(mix="cutmix")),
 "T06": ("B augmentation", dict(mix="mixup")),
 "T07": ("C loss", dict(loss="ls", label_smoothing=0.1)),
 "T08": ("C loss", dict(loss="focal", focal_gamma=2.0)),
 "T09": ("C loss", dict(loss="ce_weighted", class_weight_beta=0.0)),
 "T10": ("D sampler", dict(sampler="balanced")),
 "T11": ("F EMA", dict(ema_decay=0.995)),
 "T12": ("E LR", dict(lr_head=1e-4)),
}
for k, (ax_, o) in T.items():
    E.run_resume(cfg(exp_id=k, backbone=BEST, seed=0, **o))
Tdf = E.collect_runs(PATHS["out_dir"], "T")
Tdf["delta_vs_T00"] = Tdf.val_macro_f1 - Tdf.loc[Tdf.exp_id == "T00", "val_macro_f1"].iloc[0]
Tdf["axis"] = Tdf.exp_id.map(lambda k: T[k][0]); Tdf

In [ ]:
# TODO: kết hợp các yếu tố thắng trên val, ví dụ:
COMBO = dict(mix="cutmix", loss="ls", label_smoothing=0.1, ema_decay=0.995)
E.run_resume(cfg(exp_id="T13", backbone=BEST, seed=0, **COMBO))

## Bước 3 — Suy luận (trên VAL, dùng best.pt của T13)

In [ ]:
import numpy as np, inference as I, benchmark as BM
from eval import compute_metrics
REF = cfg(exp_id="T13", backbone=BEST, seed=0, **COMBO)
dev = torch.device("cuda"); model = E.load_best(REF, dev)
va_loader = E._loader(REF, va)
V = {"I00 1-view": {"id": I.view_identity},
     "I01 flip": {"id": I.view_identity, "flip": I.view_hflip},
     "I02 5crop": {"c": lambda x: I.views_multicrop(x, 192)}}
rows = []
for name, views in V.items():
    for space in ("prob", "logit"):
        _, y, lg = E.view_logits(model, va_loader, dev, views)
        p = I.aggregate_views(list(lg.values()), space)
        m_ = compute_metrics(y, p.argmax(1), p)
        rows.append({"method": name, "space": space, "macro_f1_val": m_["macro_f1"], "top1_val": m_["top1"], "ece_val": m_["ece"]})
Inf = pd.DataFrame(rows); Inf

In [ ]:
# Temperature scaling (T khớp trên val; ECE val trước/sau) + độ trễ
_, y, lg = E.view_logits(model, va_loader, dev, {"id": I.view_identity})
z = lg["id"]; Tt = I.fit_temperature(z, y); print("T =", Tt)
for nm, p in (("trước", I._softmax(z)), ("sau", I.apply_temperature(z, Tt))):
    print("ECE val", nm, compute_metrics(y, p.argmax(1), p)["ece"])
lat = [BM.latency_report(model, b, 224, dt) for b in (1, 32) for dt in ("fp32", "amp", "fp16")]
fused = I.fuse_conv_bn(model); lat += [{**BM.latency_report(fused, 1, 224, "fp32"), "fuse_bn": True}]
Lat = pd.DataFrame(lat); Lat

In [ ]:
# Dò độ phân giải kiểm tra (I04)
for s_ in (224, 256, 288):
    _, y, lg = E.view_logits(model, E._loader(REF, va, s_), dev, {"id": I.view_identity})
    p = I._softmax(lg["id"]); print(s_, compute_metrics(y, p.argmax(1), p)["macro_f1"])
# TODO ensemble (I05): cần >= 2 checkpoint; trung bình I.ensemble_probs([...]) trên val

## Bước 4 — Chung kết: ≥ 3 seed, test đúng một lần mỗi seed

In [ ]:
FINAL_VIEWS = {"id": I.view_identity, "flip": I.view_hflip}   # TODO chốt từ Bước 3 (val)
SPACE = "prob"
for seed in (0, 1, 2):
    c = cfg(exp_id="F01", backbone=BEST, seed=seed, **COMBO)          # huấn luyện KHÔNG ghi test
    E.run_resume(c); E.finalize(c, FINAL_VIEWS, SPACE)                # finalize chạy test đúng 1 lần
    E.run_resume(cfg(exp_id="T00", backbone=BEST, seed=seed, save_test_predictions=True))  # mốc

In [ ]:
P = PATHS["pred_dir"]; EV = f"{CODE}/eval.py"; LB = LABELS_DIR
!python {EV} score --pred "{P}/F01_seed*_test.csv" --test-csv {LB}/test_subset0.csv --labels {LB}/labels.csv --tag F01 --out {OUT}/eval_out
!python {EV} score --pred "{P}/T00_seed*_test.csv" --test-csv {LB}/test_subset0.csv --labels {LB}/labels.csv --tag T00 --out {OUT}/eval_out
!python {EV} grade --final "{P}/F01_seed*_test.csv" --baseline "{P}/T00_seed*_test.csv" --uncal "{P}/F01uncal_seed*_test.csv" --final-val "{P}/F01_seed*_val.csv" --test-csv {LB}/test_subset0.csv --labels {LB}/labels.csv --out {OUT}/eval_out

## Bước 5 — results.xlsx

In [ ]:
Final = E.collect_runs(PATHS["out_dir"], "F01")
E.write_xlsx({"Backbones": B, "Training": Tdf, "Inference": Inf, "Final": Final, "Latency": Lat,
              "Summary": pd.concat([B, Tdf]).sort_values("val_macro_f1", ascending=False).head(10)},
             f"{OUT}/results.xlsx")
# TODO: sheet PerClass từ {OUT}/eval_out; chép predictions/, curves/, results.xlsx về submissions/...